# Header

In [1]:
# .libPaths(c("/n/data1/dfci/surgonc/mittendorf/lab/chelsea/packages/miniconda3/envs/scrna_new/lib/R/library", .libPaths()))
# library(sceasy)

Loading required package: reticulate

Warning message:
“package ‘reticulate’ was built under R version 4.4.3”


In [3]:
library(Seurat)
source('functions.R')
setwd('../')
#library(Azimuth)
#library(reticulate) # Required for sceasy to interface with Python
#use_condaenv('scrna_new', required=T)

In [2]:
library(future)
options(future.globals.maxSize = 180 * 1024 * 1024 * 1024)

In [15]:
meta = read.table('data/01_meta/meta_pt.csv', sep=',', header=T) %>% arrange(dataset_name)
meta

X,sampleid,dataset_name,batch,viability_pct,group,race,age,diabetes,obesity
<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<chr>,<chr>
9,1-H,Pool137_10,1,88,HRL,NHW,62.3,N,N
24,19-I,Pool137_11,1,92,BC,NHW,48.0,N,N
1,30-D,Pool137_12,1,92,DCIS,NHW,78.0,N,N
10,43-H,Pool137_14,1,91,HRL,NHW,42.5,N,Y
25,45-I,Pool137_15,1,91,BC,NHW,67.0,N,N
26,56-I,Pool137_16,1,92,BC,NHB,60.0,N,Y
27,64-I,Pool137_17,1,90,BC,NHB,54.0,N,Y
2,36-D,Pool137_18,1,93,DCIS,NHB,39.0,N,N
28,37-I,Pool137_19,1,93,BC,NHW,79.0,N,N


In [16]:
length(list.files('data/02_preproc/seurat_tmp/'))

[1] 56

In [25]:
az_thres = .9

ct = 0
for (ds in meta$dataset_name){

    ct = ct + 1

    obj = readRDS(glue('data/02_preproc/seurat_tmp/{ds}.rds'))
    obj <- RenameCells(obj, add.cell.id = ds)
    obj@meta.data$dataset_name <- ds
    
    obj = subset(x = obj, subset = nFeature_RNA > 1000) # refine min genes to 1000

    obj = subset(x = obj, 
             subset = predicted.celltype.l1.score >= az_thres)
    obj = subset(x=obj, 
             subset = predicted.celltype.l1 != 'other')

    qs::qsave(obj, 
              file = glue("data/02_preproc/seurat_filtered/{ds}.rds")) 
    
    if (ct == 1) {mg = obj} else {mg = merge(mg, obj)}
    print(glue('{ct}, {ds}, {dim(obj)[1]}, {dim(mg)[1]}'))
    
    rm(obj); gc()  # Free memory after saving
    # if (ct > 2) break
}

1, Pool137_10, 21187, 21187
2, Pool137_11, 22132, 22857
3, Pool137_12, 20664, 23221
4, Pool137_14, 21916, 23765
5, Pool137_15, 21037, 23965
6, Pool137_16, 21274, 24183
7, Pool137_17, 22724, 24688
8, Pool137_18, 22503, 25004
9, Pool137_19, 22226, 25198
10, Pool137_20, 20430, 25321
11, Pool137_21, 23328, 25711
12, Pool137_22, 22811, 25870
13, Pool137_23, 21897, 25971
14, Pool137_24, 22050, 26060
15, Pool137_25, 22303, 26144
16, Pool137_26, 22320, 26264
17, Pool137_27, 22030, 26341
18, Pool137_28, 22638, 26446
19, Pool137_29, 22811, 26549
20, Pool137_30, 22501, 26624
21, Pool137_31, 22107, 26673
22, Pool137_32, 22544, 26814
23, Pool137_33, 22247, 26876
24, Pool137_34, 22566, 26945
25, Pool137_35, 23320, 27032
26, Pool137_36, 22298, 27068
27, Pool137_6, 22063, 27096
28, Pool137_7, 21616, 27112
29, Pool137_8, 21993, 27144
30, Pool137_9, 21832, 27175
31, Pool138_10, 22284, 27223
32, Pool138_11, 22834, 27287
33, Pool138_12, 22046, 27318
34, Pool138_13, 23690, 27466
35, Pool138_14, 23211, 2754

In [29]:
qs::qsave(mg, 
          file = glue("data/02_preproc/seurat_filtered/merge.qs")) 

In [27]:
dim(mg)

[1]  27879 444793

In [32]:
table(mg@meta.data$predicted.celltype.l1)


      B   CD4 T   CD8 T      DC    Mono      NK other T 
  53188  168410   56566    8634  111151   38197    8647 